# held-out expression PFC benchmark

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Score the four frozen GW20 forecasts (Figure 5C)

For a fresh score, run once with PFC_MOSCOT_NATIVE_PHASE=prepare (GPU only if a native forecast must be generated), then with PFC_MOSCOT_NATIVE_PHASE=score. If the frozen score CSV already exists, skip this section and run the figure cell.

In [ ]:
# QIAN PFC | GW20 retrospectively frozen, held-out 300-gene composition benchmark
# -----------------------------------------------------------------------------
# STUDY QUESTION: Which frozen *end-to-end* pipeline best predicts the cell-level
# JOINT DISTRIBUTION of 300-gene compositions at GW20, using GW15/GW22 data?
#
# Primary model choices, fixed here (NOT selected by GW20 outcomes):
#   Zoetrope: five-section fit; GW15 Tissue 2-only frozen source forecast.
#   STORIES: original native transform + frozen gene-share decoder.
#   stVCR: original evolution + native AE decoder.
#   moscot: moscot==0.5.2 OWN OT-pair interpolation (_interpolate_gex_with_ot),
#           applied to native logCP10k gene features, followed by gene shares.
#           No custom pair sampler, no hand-written time interpolation,
#           no PCA inverse decoder.
# Training budgets DIFFER by design; this is an end-to-end pipeline comparison.
# Model predictions are unchanged. No retraining or GW20-dependent calibration.
#
# TWO-PHASE USAGE: run PHASE='prepare' on a GPU with moscot==0.5.2;
# native moscot is fitted on frozen GW15+GW22 training input and its prediction
# is frozen. Then PHASE = os.environ.get('PFC_MOSCOT_NATIVE_PHASE', 'score').lower() in a CPU session reads GW20 for evaluation.
# Neither preparation nor moscot interpolation ever reads GW20.
# GW20 HAS BEEN EXAMINED in earlier exploratory work: this is a retrospective
# frozen analysis, not an untouched external confirmatory study.
#
# Primary statistic: biased empirical Gaussian-kernel MMD (not MMD^2), evaluated
# over ALL 4096 forecast cells against 1024 cells/section * 4 = 4096 true cells.
# Test reference: each GW20 section is represented equally; not donor-equal.
# Fixed training-only RBF kernel width from the original benchmark manifest.
# Additional: per-section, donor-balanced, Hellinger-share, 0.5x/2x bandwidth,
# 20 paired 1024-cell resamples (sampling stability only), 99-quantile gene-share
# RMSE, no-development / random interpolation controls, and frozen moscot
# adapter sensitivities in a SEPARATE table (not competing primary methods).
#
# No matched-cell claim, absolute-count claim, or tissue-specific forecast claim.
# IMPORTANT: This uses moscot's ORIGINAL CODE PATH for OT-pair interpolation,
# but not the public compute_interpolated_distance metric itself: that is a
# scalar PCA-space Wasserstein score, NOT a full 300-gene-share forecast.
# Native function is a PRIVATE API, so version pin and signature checks matter.

from pathlib import Path
import os, json, hashlib, gc, math
import numpy as np
import pandas as pd
import h5py
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# --------------------- FIXED PROTOCOL ---------------------
PHASE = os.environ.get('PFC_MOSCOT_NATIVE_PHASE', 'score').lower()
assert PHASE in ('prepare', 'score')
SEED = 20261027
N_PRED, N_TRUTH, N_POOL_PER_SECTION, N_REP_DRAW, N_REPS, G = 4096, 2048, 1024, 1024, 20, 300
ALPHA = 5.0 / 7.0
BW_FACTORS = (0.5, 1.0, 2.0)
METHODS = ('Zoetrope', 'STORIES', 'stVCR', 'moscot')
BASELINES = ('GW15 unchanged', 'GW22 unchanged',
             'Random-pair linear shares', 'GW15/GW22 mixture')
COLORS = {'Zoetrope':'#D62728', 'STORIES':'#4C78A8',
          'stVCR':'#F58518', 'moscot':'#54A24B'}
TEST_SECTIONS = (
    ('GW20 Tissue 1', 'FB080', 'F1', 665204),
    ('GW20 Tissue 2', 'FB080', 'F2a', 596754),
    ('GW20 Tissue 3', 'FB080', 'F2b', 322648),
    ('GW20 Tissue 4', 'FB121', 'F1', 553738),
)
TRAIN_SECTIONS = (
    ('gw15.h5ad', 'UMB1117', 'F1b', 410492),
    ('gw22.h5ad', 'FB123', 'F2', 439986),
)
PROJECT = Path(os.getenv('PFC_PROJECT',
    os.environ["STP_PROJECT"]))
DATA = Path(os.getenv('PFC_DATA_ROOT',
    os.environ["QIAN_PFC_DATA_ROOT"]))
EXP = PROJECT/'experiments/qian_pfc_publication_4way_20261024'
RUN = EXP/'GW20_heldout_fourway_Fisher_flow_vs_native_v1'
TI = PROJECT/'experiments/qian_pfc_gw20_allheldout_gw15_gw22_five_tissue_v1/CPU_FULL'
PRED_DIR = RUN/'frozen_predictions'
ORIG_FREEZE = RUN/'PREDICTIONS_FROZEN_BEFORE_GW20.json'
OLD_EVAL = RUN/'evaluation_gene_share_v1'
TI_DIR = RUN/'evaluation_source_GW15_Tissue2_only_FiveSection_TI_v1'
TI_FREEZE = TI_DIR/'PREDICTION_FROZEN_BEFORE_GW20.json'
MOS_DIR = RUN/'evaluation_moscot_native_localPCA30_gene_share_MMD_v1'
OUT = RUN/'GW20_NATIVE_MOSCOT_INTERPOLATOR_GENE_SHARE_v1'
PREP_RECEIPT = OUT/'PRE_GW20_PROTOCOL_AND_HASHES.json'
TRAIN_CONTROLS = OUT/'TRAIN_ONLY_CONTROL_FORECASTS.npz'
# Newly trained and frozen moscot forecast; kept separate from all historical files.
NATIVE_FORECAST = OUT/'FROZEN_moscot_native_interpolator_logCP10k_gene_shares.npz'
NATIVE_RECEIPT = OUT/'MOSCOT_NATIVE_INTERPOLATION_FORECAST_RECEIPT.json'
NATIVE_FACTORS = OUT/'MOSCOT_NATIVE_FGW_QRG'
MOS_TRAIN = EXP/'moscot_gw15_gw22_STRICT_GW20_HELDOUT_AUTHOR_TUTORIAL_GPU'
MOS_TRAIN_INPUT = MOS_TRAIN/'moscot_input_gw15_gw22_train_only.h5ad'
MOS_TRAIN_COMPLETE = MOS_TRAIN/'FIT_COMPLETE.json'

PRIMARY_FILES = {
    'Zoetrope': TI_DIR/'FROZEN_SpatialTI_source_GW15_Tissue2.npz',
    'STORIES': PRED_DIR/'stories.npz',
    'stVCR': PRED_DIR/'stvcr.npz',
    'moscot': OUT/'FROZEN_moscot_native_interpolator_logCP10k_gene_shares.npz',
}
# These are EXPLORATORY adapters, not independent moscot-trained methods.
OPTIONAL_MOSCOT = {
    'moscot original sampled-share adapter': PRED_DIR/'moscot.npz',
    'moscot previous PCA30 inverse adapter': MOS_DIR/'FROZEN_moscot_native_localPCA30_shares.npz',
    'moscot barycenter shares': PRED_DIR/'moscot_barycentric_shares.npz',
    'moscot barycenter logCP10k': PRED_DIR/'moscot_barycentric_logCP10k.npz',
}

# --------------------- HELPERS ---------------------
def need(p):
    p = Path(p)
    if not p.is_file():
        raise FileNotFoundError(f'Required frozen input missing: {p}')
    return p

def sha256(p):
    h = hashlib.sha256()
    with need(p).open('rb') as f:
        for block in iter(lambda: f.read(4<<20), b''): h.update(block)
    return h.hexdigest()

def index_hash(idx):
    return hashlib.sha256(np.asarray(idx, dtype='<i8').tobytes()).hexdigest()

def write_once_json(path, obj):
    path = Path(path)
    payload = json.dumps(obj, sort_keys=True, indent=2, allow_nan=False) + '\n'
    if path.exists():
        previous = json.loads(path.read_text())
        location_fields = {'data_root', 'reference_bandwidth_source'} if path == PREP_RECEIPT else set()
        if ({k:v for k,v in previous.items() if k not in location_fields} !=
                {k:v for k,v in obj.items() if k not in location_fields}):
            raise RuntimeError(f'Immutable receipt changed; use a NEW OUT directory: {path}')
        print('VERIFIED unchanged receipt:', path)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('x') as f: f.write(payload)
        print('CREATED receipt:', path)

def write_once_csv(path, table):
    path = Path(path)
    if path.exists():
        pd.testing.assert_frame_equal(pd.read_csv(path), table,
                                      check_dtype=False, check_exact=False,
                                      rtol=1e-7, atol=1e-9)
        print('VERIFIED score CSV:', path)
    else:
        with path.open('x') as f: table.to_csv(f, index=False)
        print('SAVED CSV:', path)

def validate_shares(x, label, n=N_PRED):
    x = np.asarray(x, np.float64)
    if x.shape != (n,G) or not np.isfinite(x).all() or (x<0).any():
        raise ValueError(f'{label}: invalid shares {x.shape}')
    if not np.allclose(x.sum(1), 1., rtol=0., atol=1e-5):
        raise ValueError(f'{label}: gene shares must sum to one')
    return x

def load_pred(path, genes, label, check_training=None):
    with np.load(need(path), allow_pickle=False) as z:
        if z['genes'].astype(str).tolist() != genes:
            raise ValueError(f'{label}: wrong gene order')
        x = validate_shares(z['shares'], label)
        md = json.loads(str(z['meta_json'].item())) if 'meta_json' in z.files else {}
    if not np.isclose(float(md.get('target_t_normalized', np.nan)), ALPHA):
        raise ValueError(f'{label}: wrong target time or absent metadata')
    if check_training is not None and int(md.get('training_sections', -1)) != check_training:
        raise ValueError(f'{label}: incorrect training-section count')
    return x, md

def section_indices(path, sample, region, expected_count, genes):
    ad = sc.read_h5ad(need(path), backed='r')
    try:
        if ad.raw is None or ad.raw.var_names.astype(str).tolist() != genes:
            raise ValueError(f'Original raw gene order differs: {path}')
        ids = np.flatnonzero((ad.obs['sample'].astype(str).to_numpy() == sample) &
                             (ad.obs['region'].astype(str).to_numpy() == region))
        if len(ids) != expected_count:
            raise RuntimeError(f'{path.name} {sample}/{region}: expected {expected_count}, got {len(ids)}')
        return ids
    finally:
        ad.file.close(); del ad; gc.collect()

def selected_raw_counts(path, indices):
    """Read arbitrary original raw/X rows from H5AD dense or CSR without backed fancy indexing."""
    idx = np.asarray(indices, np.int64)
    assert idx.ndim==1 and len(idx) and (idx>=0).all() and len(np.unique(idx))==len(idx)
    order=np.argsort(idx); ordered=idx[order]
    with h5py.File(need(path), 'r') as h:
        mat = h['raw/X']
        if isinstance(mat, h5py.Dataset):
            result = np.empty((len(idx),G), np.float64)
            result[order] = np.asarray(mat[ordered,:], np.float64)
        else:
            enc = mat.attrs.get('encoding-type','')
            if isinstance(enc,bytes): enc=enc.decode()
            if enc != 'csr_matrix':
                raise RuntimeError(f'Expected CSR raw/X; found {enc}. Adapt reader explicitly.')
            shape = tuple(map(int,mat.attrs['shape']))
            assert shape[1]==G and ordered.max()<shape[0]
            # Reads 4k selected row intervals only; no full sparse matrix materialization.
            ptr = mat['indptr']; cols=mat['indices']; data=mat['data']
            a=np.asarray(ptr[ordered], np.int64); b=np.asarray(ptr[ordered+1], np.int64)
            result=np.zeros((len(idx),G), np.float64)
            for k,(lo,hi) in enumerate(zip(a,b)):
                if hi>lo:
                    g=np.asarray(cols[int(lo):int(hi)], np.int32)
                    v=np.asarray(data[int(lo):int(hi)], np.float64)
                    np.add.at(result[order[k]], g, v)
    if result.shape!=(len(idx),G) or not np.isfinite(result).all() or (result<0).any():
        raise RuntimeError(f'Invalid raw counts in {path}')
    return result

def to_shares(x):
    x=np.asarray(x,np.float64)
    assert x.ndim==2 and x.shape[1]==G and np.isfinite(x).all() and (x>=0).all()
    total=x.sum(1,keepdims=True)
    if (total<=0).any(): raise ValueError('Zero-library cells: do not silently impute')
    return x/total

def sqdist(a,b):
    a=np.asarray(a,np.float64); b=np.asarray(b,np.float64)
    return np.maximum((a*a).sum(1)[:,None]+(b*b).sum(1)[None,:]-2.*(a@b.T),0.)

def median_bw(x):
    z=sqdist(x,x)
    v=float(np.median(z[np.triu_indices(len(z),1)]))
    if not np.isfinite(v) or v<=0: raise RuntimeError('Invalid training-only kernel bandwidth')
    return v

def kernel(x,y,sigma2,block=512):
    """Compute RBF Gram matrix, blockwise, float64 dot product and float32 storage."""
    x=np.asarray(x,np.float64); y=np.asarray(y,np.float64)
    k=np.empty((len(x),len(y)),dtype=np.float32)
    yn=(y*y).sum(1)[None,:]
    for i in range(0,len(x),block):
        a=x[i:i+block]
        d2=np.maximum((a*a).sum(1)[:,None]+yn-2*a@y.T,0)
        k[i:i+len(a)]=np.exp(-d2/(2.*sigma2)).astype(np.float32)
    return k

def mmd_from_k(kxx,kyy,kxy):
    v=(float(kxx.mean(dtype=np.float64))+float(kyy.mean(dtype=np.float64))-
       2.*float(kxy.mean(dtype=np.float64)))
    return math.sqrt(max(0.,v))

def paired_resampling(kxx,kyy,kxy,n_reps=N_REPS,n=N_REP_DRAW):
    # The SAME fixed indices across all methods, used only for Monte Carlo stability.
    res=[]
    for rep in range(n_reps):
        rng=np.random.default_rng(SEED+400000+rep*137)
        pi=rng.choice(kxx.shape[0],n,replace=False)
        qi=rng.choice(kyy.shape[0],n,replace=False)
        a=kxx[np.ix_(pi,pi)];b=kyy[np.ix_(qi,qi)];c=kxy[np.ix_(pi,qi)]
        res.append(mmd_from_k(a,b,c))
    return np.asarray(res)

def quantile_rmse(pred, target):
    q=np.linspace(.01,.99,99)
    p=np.quantile(pred,q,axis=0);y=np.quantile(target,q,axis=0)
    return float(np.sqrt(np.mean((p-y)**2)))

def save_figure(fig,stem):
    for ext in ('svg','pdf','png'):
        path=OUT/f'{stem}.{ext}'
        if path.exists():
            print('EXISTS (not overwritten):',path)
        else:
            fig.savefig(path,dpi=450 if ext=='png' else None,
                        bbox_inches=None,pad_inches=0,facecolor='white')
            print('SAVED:',path,'bytes:',path.stat().st_size)


# --------------------- NATIVE MOSCOT FORECAST, TRAINING ONLY ---------------------
# The organogenesis tutorial itself does not forecast a missing time point.
# Moscot's public compute_interpolated_distance() does use the package's own
# `_interpolate_gex_with_ot` routine, but compares features in its local PCA
# (30-D) transport-cost space. To score in common 300-gene share space we call
# THE SAME native interpolation routine on the original 300 logCP10k genes,
# and then apply expm1 + per-cell share normalization. This is an explicit
# evaluation-space adaptation; it is not the native PCA-space Wasserstein score.
# Source: https://github.com/theislab/moscot/blob/main/src/moscot/problems/time/_mixins.py
# No custom OT-pair sampler, no bespoke interpolation arithmetic.
# As the old FIT_COMPLETE saved only Q/R/g and no reusable solved moscot object,
# repeat the author's 2-age training to obtain an honest native API call.
# All model and adapter choices fixed independently of GW20.

def generate_native_moscot_forecast():
    from importlib.metadata import version
    import inspect, platform, time, gc
    # Must be set BEFORE the first JAX import in a fresh GPU kernel.
    if 'jax' not in __import__('sys').modules:
        os.environ['JAX_PLATFORMS']='cuda'
        os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
    import scipy.sparse as sp
    import jax
    from moscot.problems.spatiotemporal import SpatioTemporalProblem

    assert version('moscot')=='0.5.2', ('Required original moscot=0.5.2',version('moscot'))
    assert jax.devices() and all(d.platform=='gpu' for d in jax.devices()), \
        f'GPU JAX required for native moscot fit: {jax.devices()}'
    assert NATIVE_FORECAST.parent == OUT
    need(MOS_TRAIN_INPUT)
    previous=json.loads(need(MOS_TRAIN_COMPLETE).read_text())
    assert previous['pairs'][0]['shape']==[410492,439986], 'Training source cohort unexpectedly changed'
    if 'input_sha256' in previous:
        assert previous['input_sha256']==sha256(MOS_TRAIN_INPUT), 'Original training-only H5AD changed'
    print('Native moscot interpolation is being recomputed with moscot=0.5.2 on training-only data.')
    print('A new full FGW refit is intentional: original runner saved Q,R,g but not a solved moscot object.')
    tic=time.perf_counter()
    adata=sc.read_h5ad(MOS_TRAIN_INPUT)
    assert adata.n_obs==850478 and adata.n_vars==300,adata.shape
    assert adata.var_names.astype(str).tolist()==GENES, 'Training genes do not match frozen 300-gene panel'
    time_col=adata.obs['time'].astype(float).to_numpy()
    assert np.array_equal(np.unique(time_col),[15.,22.])
    assert (time_col==15.).sum()==410492 and (time_col==22.).sum()==439986
    assert 'spatial' in adata.obsm and adata.obsm['spatial'].shape==(850478,2)
    if sp.issparse(adata.X):
        assert np.isfinite(adata.X.data).all() and (adata.X.data>=0).all()
    else:
        assert np.isfinite(adata.X).all() and (adata.X>=0).all()
    print('STRICT HOLDOUT: data matrix has GW15 and GW22 only; no GW20 accessed.')
    assert 'xy_callback' in inspect.signature(SpatioTemporalProblem.prepare).parameters, 'moscot version API changed'
    moscot_problem = SpatioTemporalProblem(adata).prepare(
        time_key='time',spatial_key='spatial',joint_attr=None,
        xy_callback='local-pca', normalize_spatial=True,policy='sequential',
        a=False,b=False,
    )
    assert set(moscot_problem.problems)=={(15.,22.)},set(moscot_problem.problems)
    native=moscot_problem.problems[(15.,22.)]
    assert native.xy.data_src.shape==(410492,30)
    assert native.xy.data_tgt.shape==(439986,30)
    print('Author-native FGW prepare: local PCA30, spatial normalized, two ages, uniform marginals')
    fit_tic=time.perf_counter()
    moscot_problem=moscot_problem.solve(alpha=0.5,epsilon=0.,rank=200)
    fit_sec=time.perf_counter()-fit_tic
    sol=moscot_problem.solutions[(15.,22.)]
    assert bool(sol.is_low_rank) and int(sol.rank)==200,(sol.is_low_rank,sol.rank)
    print('Native moscot fit_seconds=',round(fit_sec,3),' converged=',sol.converged)
    interpolator=getattr(moscot_problem,'_interpolate_gex_with_ot',None)
    if not callable(interpolator):
        raise RuntimeError('moscot 0.5.2 native interpolation routine unavailable; no hand-implemented fallback')
    params=inspect.signature(interpolator).parameters
    required={'number_cells','source_data','target_data','source','target',
              'interpolation_parameter','account_for_unbalancedness','batch_size','seed'}
    assert required.issubset(params), 'Native API unexpectedly changed: '+str(params)
    pair=moscot_problem.problems[(15.,22.)]
    assert pair.adata_src.n_obs==410492 and pair.adata_tgt.n_obs==439986
    src_log=pair.adata_src.X
    tgt_log=pair.adata_tgt.X
    # The default moscot compute_interpolated_distance interpolates PCA xy;
    # archive a 30-PC output too, to expose exactly what differs in the primary
    # full-gene expression evaluation adapter. Both calls use the SAME seed.
    kws=dict(number_cells=N_PRED,source=15.,target=22.,
             interpolation_parameter=ALPHA,account_for_unbalancedness=False,
             batch_size=256,seed=SEED+888)
    print('Calling moscot native PCA30 interpolation (archive only)')
    pca_interp=interpolator(source_data=pair.xy.data_src,
                            target_data=pair.xy.data_tgt,**kws)
    print('Calling SAME moscot interpolation routine on original 300 logCP10k genes')
    # moscot 0.5.2's private interpolator calls len(source_data). SciPy sparse
    # matrices reject len(), although source_data.shape[0] is well-defined.
    # Give the native method an array-like view exposing row count and dense
    # *selected rows only*. The native pair sampler and interpolation are unchanged.
    class NativeSparseRowView:
        def __init__(self, matrix):
            self.matrix = matrix
            self.shape = matrix.shape
        def __len__(self):
            return self.shape[0]
        def __getitem__(self, key):
            subset = self.matrix[key]
            return subset.toarray() if sp.issparse(subset) else np.asarray(subset)

    source_for_native = (NativeSparseRowView(src_log)
                         if sp.issparse(src_log) else src_log)
    target_for_native = (NativeSparseRowView(tgt_log)
                         if sp.issparse(tgt_log) else tgt_log)
    assert len(source_for_native) == 410492 and len(target_for_native) == 439986
    log_interp=interpolator(source_data=source_for_native,
                            target_data=target_for_native,**kws)
    if sp.issparse(log_interp):log_interp=log_interp.toarray()
    if sp.issparse(pca_interp):pca_interp=pca_interp.toarray()
    log_interp=np.asarray(log_interp,dtype=np.float64)
    pca_interp=np.asarray(pca_interp,dtype=np.float32)
    assert log_interp.shape==(N_PRED,G) and pca_interp.shape==(N_PRED,30)
    assert np.isfinite(log_interp).all() and (log_interp>=-1.e-5).all()
    # This step is NOT interpolation: it merely converts logCP10k to per-gene
    # shares, the precommitted common evaluator space.
    exp=np.expm1(np.maximum(log_interp,0.))
    shares=to_shares(exp)
    assert shares.shape==(N_PRED,G)
    info={
      'method':'moscot', 'model':'SpatioTemporalProblem',
      'moscot_version':version('moscot'),'training_sections':2,
      'train_n_cells':850478,'source':'GW15 Tissue 2, author-native sampled OT coupling',
      'target_t_normalized':ALPHA,'source_time':15.,'target_time':22.,
      'native_interpolator':'moscot.problems.time.TemporalMixin._interpolate_gex_with_ot',
      'native_interpolator_called':True,'new_author_native_fit':True,
      'fit_solution_converged':bool(sol.converged),
      'interpolation_features':'source/target original logCP10k 300 genes; NOT local PCA30',
      'native_default_30PC_output_also_archived':True,
      'interpolation_rule':'native moscot OT-pair sample and linear feature interpolation',
      'post_processing':'expm1(interpolated logCP10k), divide by 300-gene total',
      'inverse_PCA':False,'intermediate_GW20_accessed_before_freeze':False,
      'training_data_sha256':sha256(MOS_TRAIN_INPUT),
      'fit_parameters':{'alpha':0.5,'epsilon':0.,'rank':200,
                        'local_pca_n':30,'uniform_marginals':True},
      'seed':SEED+888,'fit_seconds':float(fit_sec),'host':platform.node(),
      'important_limit':'Native method but original gene-expression inputs supplied for common space; NOT default PCA-space public Wasserstein validation.'
    }
    # Save the forecast FIRST so subsequent QC cannot suppress file creation.
    OUT.mkdir(parents=True,exist_ok=True)
    with NATIVE_FORECAST.open('xb') as f:
        np.savez_compressed(f,shares=shares.astype(np.float32),genes=np.asarray(GENES),
          native_pca30_interpolation=pca_interp,
          meta_json=np.array(json.dumps(info,sort_keys=True)))
    print('SAVED NATIVE MOSCOT FORECAST:',NATIVE_FORECAST, NATIVE_FORECAST.stat().st_size,'bytes',flush=True)
    # Archive the exact newly learned native low-rank transport, not a dense
    # n_source x n_target coupling (~700 GB). This protects scientific
    # reproducibility if moscot or its GPU backend later changes.
    NATIVE_FACTORS.mkdir(exist_ok=True)
    native_out=getattr(sol,'_output',None)
    if native_out is None or not all(hasattr(native_out,x) for x in ('q','r','g')):
        raise RuntimeError('Native moscot low-rank Q/R/g unavailable after forecast was saved')
    factor_sha={}
    for field,expected in [('q',(410492,200)),('r',(439986,200)),('g',(200,))]:
        pp=NATIVE_FACTORS/(field.upper()+'.npy')
        if pp.exists():
            raise FileExistsError('Refusing to overwrite native moscot factor: '+str(pp))
        x=np.asarray(jax.device_get(getattr(native_out,field)))
        assert x.shape==expected and np.isfinite(x).all(),(field,x.shape)
        if field=='g': assert (x>0).all(), 'Native low-rank transport contains nonpositive g'
        with pp.open('xb') as fo: np.save(fo,x)
        factor_sha[field]=sha256(pp)
        print('SAVED moscot native factor:',pp,pp.stat().st_size,'bytes')
        del x
    # Separate detailed receipt; row level input provenance and native model QC.
    receipt={'forecast_sha256':sha256(NATIVE_FORECAST),
             'native_moscot_fit_newly_computed':True,
             'original_train_only_model_complete_sha256':sha256(MOS_TRAIN_COMPLETE),
             'new_native_forecast_metadata':info,
             'version_ott_jax':version('ott-jax'),'jax':version('jax'),
             'native_transport_factor_sha256':factor_sha,
             'training_not_test':True}
    write_once_json(NATIVE_RECEIPT,receipt)
    del moscot_problem,adata,sol,pair,src_log,tgt_log,pca_interp,log_interp,shares,exp
    gc.collect()

if PHASE=='prepare' and not NATIVE_FORECAST.is_file():
    # Refuse a costly refit when other frozen evaluation inputs are missing.
    for p in (MOS_TRAIN_INPUT,MOS_TRAIN_COMPLETE,ORIG_FREEZE,
              TI_DIR/'FROZEN_SpatialTI_source_GW15_Tissue2.npz',
              PRED_DIR/'stories.npz',PRED_DIR/'stvcr.npz',
              TI/'gene_names.json',OLD_EVAL/'evaluation_manifest.json'):
        need(p)
    # Validate frozen train-only gene panel before invoking native moscot.
    GENES=list(map(str,json.loads(need(TI/'gene_names.json').read_text())))
    assert len(GENES)==G
    generate_native_moscot_forecast()
if NATIVE_FORECAST.is_file():
    need(NATIVE_RECEIPT)
    _native_rec=json.loads(NATIVE_RECEIPT.read_text())
    assert _native_rec['forecast_sha256']==sha256(NATIVE_FORECAST), \
        'Native moscot forecast hash differs from its freezing receipt'
else:
    raise FileNotFoundError('Native moscot forecast missing: run PHASE=prepare on GPU FIRST: '+str(NATIVE_FORECAST))

# --------------------- PREPARE: NO GW20 READS ---------------------
if PHASE=='score':
    need(PREP_RECEIPT)
    need(TRAIN_CONTROLS)
GENES=list(map(str,json.loads(need(TI/'gene_names.json').read_text())))
assert len(GENES)==G and len(set(GENES))==G
original=json.loads(need(ORIG_FREEZE).read_text())
expected_orig={
    'Spatial TI':PRED_DIR/'spatial_ti.npz',
    'STORIES':PRED_DIR/'stories.npz',
    'stVCR':PRED_DIR/'stvcr.npz',
    'moscot':PRED_DIR/'moscot.npz',
}
for method,path in expected_orig.items():
    if original['pred_sha256'][method] != sha256(path):
        raise RuntimeError(f'Original immutable forecast changed: {method}')

preds={}
metadata={}
for method,path in PRIMARY_FILES.items():
    x,md=load_pred(path,GENES,method,5 if method=='Zoetrope' else 2)
    if method=='Zoetrope':
        if 'GW15 Tissue 2' not in str(md.get('source','')):
            raise RuntimeError('Zoetrope must use GW15 Tissue 2 ONLY as source')
        ti_receipt=json.loads(need(TI_FREEZE).read_text())
        if ti_receipt['forecast_sha256'] != sha256(path):
            raise RuntimeError('Zoetrope Tissue2 forecast does not match its freeze')
    preds[method]=x; metadata[method]=md
    print(f'PASS {method:10s}: 4096x300 frozen, GW20 at t=5/7, hash={sha256(path)[:16]}')

# Historical BW is a locked training-only value. Reuse rather than choose a
# new kernel based on the already examined GW20 results.
old_meta=json.loads(need(OLD_EVAL/'evaluation_manifest.json').read_text())
SIGMA2=float(old_meta['sigma_sq_training_only'])
assert np.isfinite(SIGMA2) and SIGMA2>0
if 'sigma_sq_training_only' in original:
    assert np.isclose(SIGMA2,float(original['sigma_sq_training_only']),rtol=1e-8,atol=0.)

OUT.mkdir(parents=True,exist_ok=True)
optional_hashes={name:sha256(path) for name,path in OPTIONAL_MOSCOT.items() if path.is_file()}

# Training-only source and endpoint samples for benchmark controls; no GW20.
if not TRAIN_CONTROLS.exists():
    train_shares=[];train_ids=[]
    for i,(file,sample,region,n_expected) in enumerate(TRAIN_SECTIONS):
        path=need(DATA/file)
        pop=section_indices(path,sample,region,n_expected,GENES)
        rng=np.random.default_rng(SEED+700+i)
        sampled=np.sort(rng.choice(pop,6144,replace=False))
        counts=selected_raw_counts(path,sampled)
        good=np.flatnonzero(counts.sum(1)>0)
        if len(good)<N_PRED: raise RuntimeError(f'{file}: too few positive-library cells')
        take=good[:N_PRED]
        train_ids.append(sampled[take]);train_shares.append(to_shares(counts[take]))
    s,d=train_shares
    rng=np.random.default_rng(SEED+709)
    perm=rng.permutation(N_PRED)
    mixed=(1.-ALPHA)*s+ALPHA*d[perm]
    # A distributional mixture (not per-cell convex interpolation).
    mix=np.concatenate([s[:round((1.-ALPHA)*N_PRED)],
                        d[:N_PRED-round((1.-ALPHA)*N_PRED)]])
    mix=mix[np.random.default_rng(SEED+710).permutation(N_PRED)]
    assert all(np.isfinite(a).all() and np.allclose(a.sum(1),1.,atol=1e-5)
               for a in (s,d,mixed,mix))
    # BW for square-root compositions, fixed using GW15/GW22 alone.
    train_band=np.vstack([s[:256],d[:256]])
    SIGMA_HELL=median_bw(np.sqrt(train_band))
    # Original source-only/control cohort IDs are recorded, but are NOT
    # assumed identical to each method's own source-cell samples.
    with TRAIN_CONTROLS.open('xb') as f:
        np.savez_compressed(f,genes=np.asarray(GENES),source=s.astype(np.float32),
            endpoint=d.astype(np.float32),random_linear=mixed.astype(np.float32),
            population_mix=mix.astype(np.float32),bw_hell=np.array(SIGMA_HELL),
            src_rows=train_ids[0],dst_rows=train_ids[1])
    print('FROZEN TRAINING-ONLY controls:',TRAIN_CONTROLS)
with np.load(TRAIN_CONTROLS,allow_pickle=False) as arr:
    assert arr['genes'].astype(str).tolist()==GENES
    SIGMA_HELL=float(arr['bw_hell'])
    controls={
        'GW15 unchanged':validate_shares(arr['source'],'GW15 unchanged'),
        'GW22 unchanged':validate_shares(arr['endpoint'],'GW22 unchanged'),
        'Random-pair linear shares':validate_shares(arr['random_linear'],'Random-pair linear shares'),
        'GW15/GW22 mixture':validate_shares(arr['population_mix'],'GW15/GW22 mixture'),
    }
    train_ids_hash={'GW15 Tissue 2':index_hash(arr['src_rows']),
                    'GW22 Tissue 2':index_hash(arr['dst_rows'])}
assert np.isfinite(SIGMA_HELL) and SIGMA_HELL>0

protocol={
    'analysis':'RETROSPECTIVE GW20 gene-share benchmark: native moscot interpolation algorithm',
    'note':'GW20 had already been inspected in exploratory work; NOT untouched confirmatory data',
    'training_ages':['GW15','GW22'],'target_age':'GW20','forecast_time':ALPHA,
    'biological_training_input':'Zoetrope 5 sections 2240575 cells; other methods 2 sections 850478 cells',
    'source_cohort':'GW15 Tissue 2 for all primary methods (individual source cell IDs not necessarily identical)',
    'target_construction':'GW20 4 sections, 2048 nonzero cells/section chosen by original 20261027 seed; primary 1024 random cells/section',
    'target_primary':'equal-weighted 4-section GW20 population, n=4096',
    'primary_methods':list(METHODS),
    'moscot_primary_adapter':'moscot 0.5.2 TemporalMixin._interpolate_gex_with_ot, original logCP10k features -> expm1 -> 300 gene shares; NEW re-fit; no inverse PCA',
    'primary_metric':'biased RBF MMD (square root), all 4096 predicted and all 4096 observed pooled cells',
    'bandwidth_squared_training_only':SIGMA2,
    'reference_bandwidth_source':str(OLD_EVAL/'evaluation_manifest.json'),
    'secondary_target':['4 separate sections','FB080/FB121 donor-balanced pooled GW20'],
    'kernel_sensitivities':[.5,2.],
    'hellinger_share_bandwidth_training_only':SIGMA_HELL,
    'resampling':'20 paired 1024-cell draws, sampling stability only, not independent biological replicates',
    'nochange_and_interpolation_controls':list(BASELINES),
    'frozen_forecast_files_sha256':{m:sha256(p) for m,p in PRIMARY_FILES.items()},
    'optional_moscot_adapter_files_sha256':optional_hashes,
    'training_control_file_sha256':sha256(TRAIN_CONTROLS),
    'train_control_row_sha256':train_ids_hash,
    'original_freeze_sha256':sha256(ORIG_FREEZE),
    'zoetrope_source_freeze_sha256':sha256(TI_FREEZE),
    'genes_sha256':hashlib.sha256('\n'.join(GENES).encode()).hexdigest(),
    'protocol_seed':SEED,
    'data_root':str(DATA.resolve()),
    'source_model_metadata':metadata,
    'moscot_native_forecast_receipt_sha256':sha256(NATIVE_RECEIPT),
}
write_once_json(PREP_RECEIPT,protocol)
if PHASE=='prepare':
    print('\nPREPARED. GW20 WAS NOT OPENED.')
    print('Locked protocol:',PREP_RECEIPT)
    print('Review the receipt; rerun SAME CELL with PFC_MOSCOT_NATIVE_PHASE=score (CPU is sufficient).')
else:
    # --------------------- SCORE: FIRST GW20 ACCESS ---------------------
    # Score files checked before reading GW20; cannot silently overwrite.
    main_path=OUT/'GW20_PRIMARY_AND_BASELINES.csv'
    if main_path.exists():
        raise FileExistsError(f'Scores already exist. Refusing a fresh GW20 rescore: {main_path}')
    print('\nFROZEN PROTOCOL VERIFIED; now opening GW20 for evaluation only.')
    gw20=need(DATA/'gw20.h5ad')
    ad=sc.read_h5ad(gw20,backed='r')
    try:
        if ad.raw is None or ad.raw.var_names.astype(str).tolist()!=GENES:
            raise RuntimeError('GW20 raw 300-gene panel differs from frozen gene order')
        sam=ad.obs['sample'].astype(str).to_numpy()
        reg=ad.obs['region'].astype(str).to_numpy()
        populations={}
        for label,s,r,n_expected in TEST_SECTIONS:
            ids=np.flatnonzero((sam==s)&(reg==r))
            if len(ids)!=n_expected:
                raise RuntimeError(f'GW20 {label}: {len(ids)} instead of {n_expected}')
            populations[label]=ids
    finally:
        ad.file.close();del ad;gc.collect()

    truth={}; row_hashes={}; pool_parts=[]; other_parts=[]
    for j,(label,s,r,n) in enumerate(TEST_SECTIONS):
        rng=np.random.default_rng(SEED+1000+j)
        chosen=np.sort(rng.choice(populations[label],2*N_TRUTH,replace=False))
        raw=selected_raw_counts(gw20,chosen)
        good=np.flatnonzero(raw.sum(1)>0)
        if len(good)<N_TRUTH: raise RuntimeError(f'{label}: not enough nonzero cells')
        keep=good[:N_TRUTH]
        selected=chosen[keep]
        row_hashes[label]=index_hash(selected)
        share=to_shares(raw[keep]);truth[label]=share
        # Random split, NOT sorted file-order split (which could be spatially biased).
        perm=np.random.default_rng(SEED+3000+j).permutation(N_TRUTH)
        pool_parts.append(share[perm[:N_POOL_PER_SECTION]])
        other_parts.append(share[perm[N_POOL_PER_SECTION:]])
        print('GW20 observed',label,share.shape,'row sha256',row_hashes[label][:16],flush=True)

    # Strict historical cohort reproduction if the original manifest exists.
    old_cohort=OLD_EVAL/'GW20_test_cohort.csv'
    if old_cohort.exists():
        old=pd.read_csv(old_cohort)
        stored=dict(zip(old['tissue'],old['selected_index_sha256']))
        if row_hashes!=stored:
            raise RuntimeError('GW20 cell cohort differs from original scores; STOP')
        print('PASS: identical previously frozen GW20 evaluation cell IDs.')

    pooled=np.vstack(pool_parts)   # exactly 1024 cells x 4 sections
    pooled_independent=np.vstack(other_parts)
    assert pooled.shape==pooled_independent.shape==(N_PRED,G)
    # Donor-balanced distribution: 2048 FB080 and 2048 FB121 cells.
    # The three FB080 sections contribute 683, 683, 682 cells.
    fb080=np.vstack([pool_parts[0][:683],pool_parts[1][:683],pool_parts[2][:682]])
    fb121=truth['GW20 Tissue 4']
    donor_balanced=np.vstack([fb080,fb121])
    assert donor_balanced.shape==(N_PRED,G)
    TARGETS={'Equal-section GW20 pooled':pooled,
             'Donor-balanced GW20 pooled':donor_balanced,
             **truth}
    all_forecasts={**preds,**controls}

    # Exact finite-sample kernel MMD: all frozen cells, no stochastic MMD draws.
    pooled_kyy=kernel(pooled,pooled,SIGMA2)
    target_kernel={'Equal-section GW20 pooled':float(pooled_kyy.mean(dtype=np.float64))}
    for t,y in TARGETS.items():
        if t!='Equal-section GW20 pooled':
            target_kernel[t]=float(kernel(y,y,SIGMA2).mean(dtype=np.float64))
    print('Computed fixed true-reference kernel means.')
    main=[];draw_records=[];sensitivity=[]
    for method,p in all_forecasts.items():
        print('SCORING',method,flush=True)
        kxx=kernel(p,p,SIGMA2)
        kxx_mean=float(kxx.mean(dtype=np.float64))
        for label,y in TARGETS.items():
            kxy=kernel(p,y,SIGMA2)
            score=math.sqrt(max(0.,kxx_mean+target_kernel[label]-2.*float(kxy.mean(dtype=np.float64))))
            rec=dict(target=label,method=method,
                     model_or_control='model' if method in METHODS else 'simple control',
                     MMD_gene_share=score,n_forecast=len(p),n_truth=len(y),
                     mean_share_RMSE=float(np.sqrt(np.mean((p.mean(0)-y.mean(0))**2))),
                     gene_share_quantile_RMSE=quantile_rmse(p,y))
            main.append(rec)
            if label=='Equal-section GW20 pooled' and method in METHODS:
                draws=paired_resampling(kxx,pooled_kyy,kxy)
                draw_records.extend(dict(method=method,replicate=i,MMD_gene_share=float(val))
                                    for i,val in enumerate(draws))
            del kxy
        del kxx
        if method in METHODS:
            # Prespecified BW sensitivity on PRIMARY pooled target only.
            for factor in (0.5,2.0):
                sig=SIGMA2*factor
                score=mmd_from_k(kernel(p,p,sig),kernel(pooled,pooled,sig),kernel(p,pooled,sig))
                sensitivity.append(dict(method=method,analysis='RBF bandwidth',
                                        setting=f'{factor:g}x',MMD_gene_share=score))
            # Hellinger-related transformation of compositional gene shares.
            score=mmd_from_k(kernel(np.sqrt(p),np.sqrt(p),SIGMA_HELL),
                             kernel(np.sqrt(pooled),np.sqrt(pooled),SIGMA_HELL),
                             kernel(np.sqrt(p),np.sqrt(pooled),SIGMA_HELL))
            sensitivity.append(dict(method=method,analysis='sqrt(gene-share) RBF',
                                    setting='training-only BW',MMD_gene_share=score))
        gc.collect()

    main=pd.DataFrame(main)
    draws=pd.DataFrame(draw_records)
    sens=pd.DataFrame(sensitivity)

    # Split-half diagnostic is a sampling reference, NOT an error floor to subtract.
    split=mmd_from_k(kernel(pooled,pooled,SIGMA2),
                     kernel(pooled_independent,pooled_independent,SIGMA2),
                     kernel(pooled,pooled_independent,SIGMA2))
    # Optional earlier moscot adapters are diagnostics ONLY, not primary models.
    adapter_rows=[]
    for name,path in OPTIONAL_MOSCOT.items():
        if name not in optional_hashes: continue
        if sha256(path)!=optional_hashes[name]:
            raise RuntimeError(f'Alternative adapter changed after protocol freeze: {name}')
        p,md=load_pred(path,GENES,name,2)
        score=mmd_from_k(kernel(p,p,SIGMA2),pooled_kyy,kernel(p,pooled,SIGMA2))
        adapter_rows.append(dict(adapter=name,target='Equal-section GW20 pooled',
                                 MMD_gene_share=score,
                                 interpretation='post-hoc sensitivity; NOT a primary comparator'))
        print('ADAPTER SENSITIVITY',name,score,flush=True)

    # Score tables are written before any plotting; existing files never overwritten.
    write_once_csv(OUT/'GW20_PRIMARY_AND_BASELINES.csv',main)
    write_once_csv(OUT/'GW20_PAIRED_SUBSAMPLING_STABILITY.csv',draws)
    write_once_csv(OUT/'GW20_KERNEL_SENSITIVITIES.csv',sens)
    write_once_csv(OUT/'GW20_OPTIONAL_MOSCOT_ADAPTERS_EXPLORATORY.csv',pd.DataFrame(
        adapter_rows,columns=['adapter','target','MMD_gene_share','interpretation']))
    write_once_csv(OUT/'GW20_EVALUATED_COHORT_HASHES.csv',pd.DataFrame([
        dict(tissue=t,observed_index_sha256=row_hashes[t]) for t in row_hashes]))
    summaries=(main[(main['target']=='Equal-section GW20 pooled') &
                    (main['method'].isin(METHODS))]
               .set_index('method').loc[list(METHODS)])
    print('\nPRIMARY: equal-section GW20 pooled full 300-gene shares (lower better)')
    print(summaries[['MMD_gene_share','mean_share_RMSE','gene_share_quantile_RMSE']]
          .to_string(float_format=lambda v:f'{v:.6f}'))
    print('Reference GW20 split-half MMD (not subtractable):',round(split,6))
    print('\nN.B. 20 resampling repetitions measure finite-cell sampling stability,')
    print('NOT donor independence; FB080 contributes 3 of the 4 sections.')
    write_once_json(OUT/'GW20_SCORE_MANIFEST.json',{
        'protocol_sha256':sha256(PREP_RECEIPT),
        'reference_gw20_split_half_mmd':split,
        'cohort_row_hashes':row_hashes,
        'n_frozen_predictions':N_PRED,
        'primary_pooled_reference_per_section':N_POOL_PER_SECTION,
        'MMD':'biased, finite-sample, Gaussian kernel, square root reported',
        'pooled_vs_section_mean':'distinct estimands; do not conflate',
        'donor_replicates':'two donors only: FB080 (3 sections), FB121 (1 section)',
        'no_validation_tuning':True,
        'historical_data_use':'GW20 was previously examined while developing evaluation protocol',
    })

    # Publication-format compact figure with model methods ONLY.
    mpl.rcParams.update({'font.family':'DejaVu Sans','font.size':10,
        'axes.titlesize':10.5,'axes.titleweight':'bold','axes.labelsize':10.3,
        'xtick.labelsize':9.1,'ytick.labelsize':9.1,
        'axes.edgecolor':'#26343D','axes.labelcolor':'#26343D','text.color':'#26343D',
        'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none',
        'figure.facecolor':'white','savefig.facecolor':'white'})
    fig=plt.figure(figsize=(3.35,3.25),facecolor='white')
    ax=fig.add_axes([.22,.265,.75,.57])
    heights=summaries['MMD_gene_share'].to_numpy()
    x=np.arange(len(METHODS))
    bars=ax.bar(x,heights,color=[COLORS[m] for m in METHODS],width=.75,edgecolor='none')
    ax.set_xticks(x)
    ax.set_xticklabels(METHODS,rotation=38,ha='right',rotation_mode='anchor',fontweight='bold')
    ax.set_ylabel('300-gene share MMD ↓')
    ax.set_title('Held-out GW20 expression',pad=8)
    ax.set_ylim(0,max(0.02,float(heights.max())*1.20))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',alpha=.16,lw=.7);ax.set_axisbelow(True)
    for b,v in zip(bars,heights):
        ax.text(b.get_x()+b.get_width()/2,v+max(.005,float(heights.max())*.035),
                f'{v:.3f}',ha='center',va='bottom',fontsize=8.9)
    save_figure(fig,'FIG_GW20_PRIMARY_POOLED_GENE_SHARE_MMD_335in')
    plt.show();plt.close(fig)
    print('\nDONE. All tables and 3 plot formats in:',OUT)

## Render Figure 5C–D

In [ ]:
# %% [ONE SELF-CONTAINED JUPYTER CELL]
# PFC publication figure: held-out expression + runtime
# Size: 2.50" wide x 4.38" tall
# No A/B panel labels. No model retraining.

from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PROJECT = Path(os.getenv(
    "STP_PROJECT",
    os.environ["STP_PROJECT"]
))

RUNTIME_CSV = PROJECT / (
    "experiments/qian_pfc_timing_three_sections_1_2_7_v1/"
    "GPU_FULL/PFC_three_age_core_training_times.csv"
)

HELDOUT_CSV = PROJECT / (
    "experiments/qian_pfc_publication_4way_20261024/"
    "GW20_heldout_fourway_Fisher_flow_vs_native_v1/"
    "GW20_NATIVE_MOSCOT_INTERPOLATOR_GENE_SHARE_v1/"
    "GW20_PRIMARY_AND_BASELINES.csv"
)

OUT = PROJECT / (
    "experiments/qian_pfc_publication_4way_20261024/"
    "stacked_runtime_and_heldout_panel"
)
OUT.mkdir(parents=True, exist_ok=True)

for path in (RUNTIME_CSV, HELDOUT_CSV):
    if not path.is_file():
        raise FileNotFoundError(path)

# ------------------------------------------------------------
# Figure configuration
# ------------------------------------------------------------
FIG_W, FIG_H = 2.50, 4.38
DPI = 450

STEM = (
    "FIG_PFC_GW20_MMD_TOP_runtime_BOTTOM_"
    "2p5x4p38_NO_LETTERS"
)

ORDER = ["Zoetrope", "STORIES", "stVCR", "moscot"]

METHOD_COLORS = {
    "Zoetrope": "#D62728",
    "STORIES":  "#4C78A8",
    "stVCR":    "#F58518",
    "moscot":   "#54A24B",
}

INK = "#26343D"
GRID = "#CFD5DB"

# ------------------------------------------------------------
# Load frozen benchmark results
# ------------------------------------------------------------
runtime = pd.read_csv(RUNTIME_CSV)
heldout = pd.read_csv(HELDOUT_CSV)

def canonical_name(name):
    s = str(name).strip().lower()
    if s in ("zoetrope", "spatial ti", "spatial-ti",
             "flow-stp", "fisher flow"):
        return "Zoetrope"
    if s == "stories":
        return "STORIES"
    if s == "stvcr":
        return "stVCR"
    if s == "moscot":
        return "moscot"
    return None

def ordered_values(frame, value_column):
    df = frame.copy()
    df["display"] = df["method"].map(canonical_name)
    df = df[df["display"].isin(ORDER)].copy()

    if df["display"].duplicated().any():
        raise ValueError("Duplicate rows for one method")

    df = df.set_index("display")
    missing = set(ORDER) - set(df.index)
    if missing:
        raise ValueError(f"Missing methods: {missing}")

    values = df.loc[ORDER, value_column].to_numpy(float)
    assert np.isfinite(values).all() and (values >= 0).all()
    return values

# Explicitly select the primary pooled GW20 target
heldout_primary = heldout[
    heldout["target"] == "Equal-section GW20 pooled"
].copy()

if "model_or_control" in heldout_primary:
    heldout_primary = heldout_primary[
        heldout_primary["model_or_control"] == "model"
    ]

runtime_vals = ordered_values(runtime, "fit_minutes")
mmd_vals = ordered_values(heldout_primary, "MMD_gene_share")

# ------------------------------------------------------------
# Publication style
# ------------------------------------------------------------
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.8,
    "axes.titlesize": 9.8,
    "axes.titleweight": "bold",
    "axes.labelsize": 9.2,
    "axes.linewidth": 0.75,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "xtick.labelsize": 8.3,
    "ytick.labelsize": 8.3,
    "xtick.color": INK,
    "ytick.color": INK,
    "text.color": INK,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.facecolor": "white",
})

# ------------------------------------------------------------
# Stacked figure
# ------------------------------------------------------------
fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor="white")

# Top: held-out expression
ax_mmd = fig.add_axes([0.23, 0.60, 0.73, 0.28])

# Bottom: runtime
ax_runtime = fig.add_axes([0.23, 0.15, 0.73, 0.28])

x = np.arange(len(ORDER))
colors = [METHOD_COLORS[m] for m in ORDER]

def format_axis(ax, title, ylabel):
    ax.set_title(
        title, pad=4,
        fontsize=9.8, fontweight="bold"
    )
    ax.set_ylabel(
        ylabel, fontsize=9.2, labelpad=3
    )

    ax.set_xlim(-0.5, len(ORDER) - 0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(
        ORDER,
        rotation=42,
        ha="right",
        rotation_mode="anchor",
        fontweight="bold"
    )

    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.grid(
        axis="y", color=GRID,
        alpha=0.65, linewidth=0.6
    )
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", length=0, pad=1)
    ax.tick_params(axis="y", length=3, width=0.7, pad=1.5)

def annotate(ax, values, labels):
    for xi, value, label in zip(x, values, labels):
        ax.annotate(
            label, (xi, value),
            xytext=(0, 2.8),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8.0,
            fontweight="bold",
            color="#5F6973"
        )

# ------------------------------------------------------------
# TOP: held-out GW20 expression
# ------------------------------------------------------------
ax_mmd.bar(
    x, mmd_vals,
    width=0.74,
    color=colors,
    edgecolor="none",
    zorder=3
)

ax_mmd.set_ylim(0, max(0.01, mmd_vals.max() * 1.20))

format_axis(
    ax_mmd,
    "Held-out expression · MMD",
    "Gene-share MMD ↓"
)

annotate(
    ax_mmd, mmd_vals,
    [f"{v:.3f}" for v in mmd_vals]
)

# ------------------------------------------------------------
# BOTTOM: GPU runtime comparison
# ------------------------------------------------------------
ax_runtime.bar(
    x, runtime_vals,
    width=0.74,
    color=colors,
    edgecolor="none",
    zorder=3
)

ax_runtime.set_ylim(
    0, max(0.5, runtime_vals.max() * 1.20)
)

format_axis(
    ax_runtime,
    "3-slice runtime",
    "Minutes ↓"
)

annotate(
    ax_runtime,
    runtime_vals,
    [
        f"{v:.2f}" if v < 1 else f"{v:.1f}"
        for v in runtime_vals
    ]
)

# No A/B panel labels

# ------------------------------------------------------------
# Save PNG, PDF, SVG
# ------------------------------------------------------------
for ext in ("png", "pdf", "svg"):
    path = OUT / f"{STEM}.{ext}"

    fig.savefig(
        path,
        dpi=DPI if ext == "png" else None,
        facecolor="white",
        bbox_inches=None,
        pad_inches=0
    )
    print("SAVED:", path)

receipt = pd.DataFrame({
    "method": ORDER,
    "heldout_GW20_MMD": mmd_vals,
    "three_slice_fit_minutes": runtime_vals
})

receipt.to_csv(
    OUT / f"{STEM}_values.csv",
    index=False
)

plt.show()
plt.close(fig)

## Optional Zoetrope GW20 forecast export

Run this section by itself after setup, before the score cell, only if the original Zoetrope forecast must be exported. Set ZOETROPE_EXPORT_PFC_HOLDOUT=1 before running this cell.

In [ ]:
if os.environ.get("ZOETROPE_EXPORT_PFC_HOLDOUT", "0") == "1":
    # Qian PFC held-out GW20: five-section trained Spatial TI, GW15 Tissue 2 source ONLY.
    # Compares with unchanged frozen STORIES, stVCR and moscot local-PCA forecasts.
    # Re-scores the EXACT original cohort and kernel; verifies unchanged baselines.
    # DOES NOT retrain, overwrite previous predictions, or use GW20 to construct forecast.

    from pathlib import Path
    import os, sys, json, hashlib, gc
    import numpy as np
    import pandas as pd
    import torch

    SEED=20261027
    N_PRED,G,N_TRUTH,N_MMD,N_REPS=4096,300,2048,1024,3
    ALPHA=5./7.
    NZ,NT,NSTEPS=384,73,128
    METHODS=['Spatial TI','STORIES','stVCR','moscot']
    COLORS={'Spatial TI':'#D62728','STORIES':'#4C78A8',
            'stVCR':'#F58518','moscot':'#54A24B'}
    PROJECT=Path(os.environ.get('PFC_PROJECT',
     os.environ["STP_PROJECT"]))
    DATA=Path(os.environ.get('PFC_DATA_ROOT',
     os.environ["QIAN_PFC_DATA_ROOT"]))
    REPO=Path(os.environ.get('FLOW_STP_REPO',
     os.environ["FLOW_STP_REPO"]))
    if not (REPO/'src/flow_stp/stage1/stage1a.py').is_file() and (PROJECT/'src/flow_stp/stage1/stage1a.py').is_file():
     REPO=PROJECT
    sys.path.insert(0,str(REPO/'src'))
    EXP=PROJECT/'experiments/qian_pfc_publication_4way_20261024'
    RUN=EXP/'GW20_heldout_fourway_Fisher_flow_vs_native_v1'
    TI=PROJECT/'experiments/qian_pfc_gw20_allheldout_gw15_gw22_five_tissue_v1/CPU_FULL'
    BASE=RUN/'evaluation_gene_share_v1'
    MOS_RUN=RUN/'evaluation_moscot_native_localPCA30_gene_share_MMD_v1'
    OLD_FREEZE=RUN/'PREDICTIONS_FROZEN_BEFORE_GW20.json'
    MOS_FREEZE=MOS_RUN/'PREDICTION_FROZEN_BEFORE_GW20.json'
    ORIG_SP=RUN/'frozen_predictions/spatial_ti.npz'
    ORIG_MOS=RUN/'frozen_predictions/moscot.npz'
    PRED={
     'STORIES':RUN/'frozen_predictions/stories.npz',
     'stVCR':RUN/'frozen_predictions/stvcr.npz',
     'moscot':MOS_RUN/'FROZEN_moscot_native_localPCA30_shares.npz',
    }
    OUT=RUN/'evaluation_source_GW15_Tissue2_only_FiveSection_TI_v1'
    NEW_SP=OUT/'FROZEN_SpatialTI_source_GW15_Tissue2.npz'
    NEW_FREEZE=OUT/'PREDICTION_FROZEN_BEFORE_GW20.json'

    def must(p):
     p=Path(p)
     if not p.is_file():raise FileNotFoundError(f'Missing required input: {p}')
     return p

    def sha(p):
     h=hashlib.sha256()
     with open(must(p),'rb') as f:
      for b in iter(lambda:f.read(4<<20),b''):h.update(b)
     return h.hexdigest()

    def save_once(path,obj):
     txt=json.dumps(obj,indent=2,sort_keys=True)+'\n'
     if path.exists():
      if json.loads(path.read_text())!=obj:raise RuntimeError(f'Provenance mismatch: {path}')
     else:path.write_text(txt)

    def load_forecast(path):
     with np.load(must(path),allow_pickle=False) as z:
      assert z['genes'].astype(str).tolist()==GENES
      p=z['shares'].astype(np.float64)
     assert p.shape==(N_PRED,G) and np.isfinite(p).all() and (p>=0).all()
     assert np.allclose(p.sum(1),1,atol=1e-5)
     return p

    # 1. Freeze/provenance checks: training-only artifacts before accessing GW20.
    GENES=list(map(str,json.loads(must(TI/'gene_names.json').read_text())))
    assert len(GENES)==G and len(set(GENES))==G
    old_manifest=json.loads(must(OLD_FREEZE).read_text())
    for name,p in {'Spatial TI':ORIG_SP,'STORIES':PRED['STORIES'],
                   'stVCR':PRED['stVCR'],'moscot':ORIG_MOS}.items():
     assert old_manifest['pred_sha256'][name]==sha(p),f'{name} checksum differs from frozen run'
    mos_manifest=json.loads(must(MOS_FREEZE).read_text())
    assert mos_manifest['prediction_sha256']==sha(PRED['moscot'])
    assert all(load_forecast(path).shape==(N_PRED,G)
               for path in [ORIG_SP,ORIG_MOS,*PRED.values()])
    OUT.mkdir(parents=True,exist_ok=True)

    # 2. Generate NEW forecast from GW15 Tissue 2 anatomical coordinates ONLY.
    if not NEW_SP.exists():
     from flow_stp.stage1.stage1a import FlowSTPModel,FlowSTPConfig
     from torch.func import jvp
     from flow_stp.stage2.dynamics import velocity_field
     must(TI/'done.json')
     stage=pd.read_csv(must(TI/'tables/stage1a_all_cells.csv.gz'),
                       usecols=['slice_id','z_stage1a'])
     counts=stage.groupby('slice_id').size().reindex(range(5)).tolist()
     assert counts==[548567,410492,508417,439986,333113],counts
     z_all=stage.z_stage1a.to_numpy(np.float64)
     sids=stage.slice_id.to_numpy()
     z_t2=z_all[sids==1]
     assert len(z_t2)==410492 and np.isfinite(z_t2).all()
     rng=np.random.default_rng(SEED+111)
     chosen_idx=np.sort(rng.choice(len(z_t2),size=N_PRED,replace=False))
     z0=z_t2[chosen_idx].copy()
     ckpt=must(TI/'model_final.pt')
     ck=torch.load(ckpt,map_location='cpu',weights_only=False)
     conf=FlowSTPConfig(**ck['model_config'])
     assert conf.num_genes==G and conf.num_slices==5
     assert conf.coordinate_decoder=='shared_time_tissue'
     model=FlowSTPModel(conf).cpu().eval()
     model.load_state_dict(ck['model_state_dict'],strict=True)
     zr=float(np.ptp(z_all));assert zr>0
     zg=np.linspace(z_all.min()-.45*zr,z_all.max()+.45*zr,NZ)
     tg=np.linspace(0.,ALPHA,NT)
     vg=np.empty((NT,NZ),dtype=np.float64)
     for j,t in enumerate(tg):
      for a in range(0,NZ,192):
       zz=torch.as_tensor(zg[a:a+192],dtype=torch.float32)
       tt=torch.full_like(zz,float(t))
       func=lambda u,w:model.fisher_flat(u,w)
       _,dz=jvp(func,(zz,tt),(torch.ones_like(zz),torch.zeros_like(zz)))
       _,dtj=jvp(func,(zz,tt),(torch.zeros_like(zz),torch.ones_like(zz)))
       v=(-(dz*dtj).sum(1)/((dz*dz).sum(1)+1e-8)).detach().numpy()
       vg[j,a:a+len(v)]=v
      if (j+1)%12==0:print(f'Fisher velocity grid {j+1}/{NT}',flush=True)
     assert np.isfinite(vg).all()
     probe=torch.as_tensor(np.quantile(z_t2,[.1,.4,.7,.9]),dtype=torch.float32)
     native=velocity_field(model.fisher_flat,probe,
                          torch.full_like(probe,.35),eps=1e-8).numpy().reshape(-1)
     approx=np.array([np.interp(z,zg,vg[np.argmin(abs(tg-.35))]) for z in probe.numpy()])
     rel=float(np.max(abs(native-approx)/(1+abs(native))))
     if rel>.08:raise RuntimeError(f'Velocity grid did not verify: {rel}')
     def velocity(z,t):
      if np.min(z)<zg[0] or np.max(z)>zg[-1]:
       raise RuntimeError('Transport left velocity domain; no clipping allowed')
      v=t/ALPHA*(NT-1);lo=min(int(np.floor(v)),NT-2);a=v-lo
      return (1-a)*np.interp(z,zg,vg[lo])+a*np.interp(z,zg,vg[lo+1])
     z=z0.copy();h=ALPHA/NSTEPS
     for k in range(NSTEPS):
      t=k*h
      k1=velocity(z,t)
      k2=velocity(z+.5*h*k1,t+.5*h)
      k3=velocity(z+.5*h*k2,t+.5*h)
      k4=velocity(z+h*k3,t+h)
      z+=h*(k1+2*k2+2*k3+k4)/6
     assert np.isfinite(z).all()
     expr=[]
     with torch.no_grad():
      for a in range(0,N_PRED,1024):
       zz=torch.as_tensor(z[a:a+1024],dtype=torch.float32)
       tt=torch.full_like(zz,ALPHA)
       expr.append(model.expression(zz,tt).numpy())
     e=np.concatenate(expr).astype(np.float64)
     assert e.shape==(N_PRED,G) and np.isfinite(e).all() and (e>=0).all()
     assert (e.sum(1)>0).all()
     p=(e/e.sum(1,keepdims=True)).astype(np.float32)
     meta={'method':'Spatial TI','source':'GW15 Tissue 2 ONLY (slice_id=1)',
           'source_sample_indices_sha256':hashlib.sha256(chosen_idx.astype('<i8').tobytes()).hexdigest(),
           'n_source_cells':410492,'training_sections':5,'train_n_cells':2240575,
           'target_t_normalized':ALPHA,'grid_v_z':NZ,'grid_v_t':NT,
           'rk4_steps':NSTEPS,'model_checkpoint_sha256':sha(ckpt),
           'max_rel_velocity_grid_error':rel,
           'mean_z_initial':float(z0.mean()),'mean_z_future':float(z.mean())}
     # Freeze without touching original five-section forecast
     np.savez_compressed(NEW_SP,shares=p,genes=np.array(GENES),
                         meta_json=np.array(json.dumps(meta,sort_keys=True)),
                         source_stage1a_row_indices=chosen_idx)
     print('FROZEN Spatial TI GW15 Tissue2 source:',NEW_SP,sha(NEW_SP))
     del model,ck,stage,e,p,z,vg;gc.collect()
    else:
     print('Reusing frozen GW15 Tissue 2 Spatial TI forecast:',NEW_SP)

    P={name:load_forecast(file) for name,file in PRED.items()}
    P['Spatial TI']=load_forecast(NEW_SP)
    CONTROL_SP=load_forecast(ORIG_SP)
    CONTROL_MOS=load_forecast(ORIG_MOS)
    new_manifest={'forecast_sha256':sha(NEW_SP),'original_model_sha256':sha(TI/'model_final.pt'),
                  'old_prediction_freeze_sha256':sha(OLD_FREEZE),
                  'native_moscot_prediction_sha256':sha(PRED['moscot']),
                  'source':'GW15 Tissue 2 (410492 cells)',
                  'model_training':'five GW15/GW22 tissue sections (unchanged)',
                  'GW20_was_not_accessed_in_prediction':True}
    save_once(NEW_FREEZE,new_manifest)